In [ ]:
%reset -f
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import os
from scipy.spatial.distance import cosine
warnings.filterwarnings('ignore')
os.chdir(r'G:\Kuangyu_Temp\Outsource\description')
# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置绘图风格
sns.set_style("whitegrid")
sns.set_palette("husl")

# 读取数据
firm_agg = pd.read_stata('summary.dta')
firm_agg = firm_agg[firm_agg['year'] == 2018]
df1 = pd.read_stata(r'G:\Kuangyu_Temp\Outsource\lenth9_18.dta')

In [ ]:
np.random.seed(42)


selected = firm_agg[(firm_agg['is_intermediary'] == 0)&(firm_agg['outsourcing_intensity']>0)]
sampled_firms = selected.sample(n = min(50000, len(selected)), random_state=42)['firm_id'].values
sampled_firms = [str(x) for x in sampled_firms]

In [17]:
df = df1[df1['firm_id'].isin(sampled_firms)]

buy = df[df['is_output'] == 0].groupby(['firm_id', 'product_id'])['v'].sum().reset_index()
buy.columns = ['firm_id', 'product_id', 'buy']

sell = df[df['is_output'] == 1].groupby(['firm_id', 'product_id'])['v'].sum().reset_index()
sell.columns = ['firm_id', 'product_id', 'sell']

prod = sell.merge(buy, on = ['firm_id', 'product_id'], how = 'left').fillna(0)
prod['outsource'] = prod[['buy', 'sell']].min(axis = 1)
prod['own'] = prod['sell'] - prod['outsource']

In [19]:
io = pd.read_stata(r'G:\Kuangyu_Temp\Outsource\io_table_lite.dta')

results = []

for firm in sampled_firms:
    fp = prod[prod['firm_id'] == firm]
    
    outsource_prod = fp[fp['outsource']>0][['product_id', 'outsource']]
    own_prod = fp[fp['own']>0][['product_id', 'own']]
    
    if len(outsource_prod) == 0 or len(own_prod) == 0:
        continue
    
    io_out = io[io['product_id'].isin(outsource_prod['product_id'])].copy()
    io_out = io_out.merge(outsource_prod, on = 'product_id')
    vec_out = io_out.groupby('product_id_input').apply(
        lambda x: (x['coefficient']*x['outsource']).sum()
    )/outsource_prod['outsource'].sum()
    
    io_own = io[io['product_id'].isin(own_prod['product_id'])].copy()
    io_own = io_own.merge(own_prod, on = 'product_id')
    vec_own = io_own.groupby('product_id_input').apply(
        lambda x: (x['coefficient']*x['own']).sum()
    )/own_prod['own'].sum()
    
    #统一维度
    all_inputs = sorted(set(vec_out.index) | set(vec_own.index))
    v1 = [vec_out.get(i, 0) for i in all_inputs]
    v2 = [vec_own.get(i, 0) for i in all_inputs]
    
    if sum(v1) >0 and sum(v2)>0:
        sim = 1-cosine(v1, v2)
        results.append({'firm_id': firm, 'cosine_similarity':sim})

In [21]:
results_df = pd.DataFrame(results)
results_df.to_stata('similarity.dta', write_index = False)